In [ ]:
# Celda 1: Importaciones
import pandas as pd
import numpy as np
import joblib
import sys
import os
import warnings
warnings.filterwarnings('ignore')

sys.path.append(os.path.abspath('..'))

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

In [ ]:
# Celda 2: Preparación de datos
df = pd.read_csv('data/raw/train.csv')

from src.features import feature_engineering
df = feature_engineering(df)

df['SalePrice'] = np.log1p(df['SalePrice'])
y = df['SalePrice']
X = df.drop(['SalePrice', 'Id'], axis=1, errors='ignore')

from src.preprocessing import create_preprocessor
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

preprocessor = create_preprocessor(numeric_features, categorical_features)
X_processed = preprocessor.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42
)

In [ ]:
# Celda 3: Entrenamiento de los 3 modelos
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

models = {
    'XGBoost': XGBRegressor(n_estimators=800, learning_rate=0.05, max_depth=6, random_state=42),
    'LightGBM': LGBMRegressor(n_estimators=800, learning_rate=0.05, max_depth=6, random_state=42, verbose=-1),
    'CatBoost': CatBoostRegressor(n_estimators=800, learning_rate=0.05, depth=6, random_state=42, verbose=False)
}

results = []

for name, model in models.items():
    print(f"Entrenando {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    y_test_orig = np.expm1(y_test)
    y_pred_orig = np.expm1(y_pred)
    
    rmsle = np.sqrt(mean_squared_log_error(y_test_orig, y_pred_orig))
    mae = mean_absolute_error(y_test_orig, y_pred_orig)
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        'Modelo': name,
        'RMSLE': round(rmsle, 5),
        'MAE ($)': round(mae, 2),
        'R²': round(r2, 4)
    })

# Mostrar resultados
comparison_df = pd.DataFrame(results)
display(comparison_df.sort_values('RMSLE'))

In [ ]:
# Celda 4: Gráfico comparativo
plt.figure(figsize=(10,6))
sns.barplot(data=comparison_df, x='Modelo', y='RMSLE')
plt.title('Comparación de RMSLE entre Modelos')
plt.ylabel('RMSLE (menor es mejor)')
plt.show()

In [ ]:
# Celda 5: Guardar el mejor modelo
best_model_name = comparison_df.loc[comparison_df['RMSLE'].idxmin(), 'Modelo']
print(f"✅ Mejor modelo: {best_model_name}")

best_model = models[best_model_name]
joblib.dump(best_model, 'models/best_model_final.pkl')
joblib.dump(preprocessor, 'models/preprocessor_final.pkl')
print("Modelos guardados como 'best_model_final.pkl'")